# 26: A print figure of one RF drift cell flight, drawn by the run-export framework

This notebook flies the Allen and Bush RF confining drift cell at one operating point and draws a three-panel print figure of the flight. Panel A shows the ring stack with example trajectories, panel B shows where the ions crossed the exit plane, and panel C shows their arrival times at that plane. Every panel is drawn from solver products by the generic run-export framework, which works for any planar or r-z deck. The impact arrivals come from `physics.stations.impact_hits`, the same function the workbench's Impact Analysis tab uses, so the printed figure and the workbench show the same flight.

The device and ion follow the campaign of `24_srig_rf_mechanism_v2.ipynb`, and the operating point is the middle value of each campaign ladder. The flight is written to a bank, and the figure is drawn from the bank, so the figure can be redrawn later without flying again.

## 1. Every setting this notebook uses is named in one cell

In [ ]:
# ---- import-origin guard (run me FIRST) --------------------------------
# THIS notebook belongs to a repo; it must run against THAT repo's
# ion_gym, not whatever `import ion_gym` happens to find (a pip-installed
# copy, or an old tree on PYTHONPATH). A version mismatch does not fail
# politely -- it surfaces mid-run as a confusing AttributeError on some
# API the stale copy predates. So: locate the repo from this notebook's
# working directory, put it FIRST on sys.path, then verify the imported
# package actually came from here -- and REFUSE with the remedy if not.
# GENERATED CELL: every notebook carries one identical copy, stamped from
# a single definition in the development tree. Edits made here are
# overwritten the next time the notebook is regenerated.
import sys
from pathlib import Path

# leading underscores here are DELIBERATE (stated reason): this
# cell is stamped into every notebook and must not collide with or
# pollute the study's own names
_here = Path.cwd().resolve()
ROOT_GUARD = next((p for p in (_here, *_here.parents)
                   if (p / "ion_gym").is_dir() and (p / "notebooks").is_dir()),
                  None)
if ROOT_GUARD is None:
    raise RuntimeError(
        f"cannot locate the ion_gym repo at or above {_here}; start the "
        f"kernel in the repo root or in a folder inside it")
if str(ROOT_GUARD) not in sys.path:
    sys.path.insert(0, str(ROOT_GUARD))

import ion_gym
_origin = Path(ion_gym.__file__).resolve().parent
if _origin.parent != ROOT_GUARD:
    raise RuntimeError(
        f"imported ion_gym v{ion_gym.__version__} from {_origin}, which is "
        f"NOT this repo ({ROOT_GUARD / 'ion_gym'}). Either the kernel "
        f"already imported a stale copy (restart the kernel and run this "
        f"cell first) or another copy shadows the repo (a pip-installed "
        f"ion_gym: `pip uninstall ion_gym`; or a stale PYTHONPATH entry). "
        f"Refusing now beats an AttributeError several cells later.")
print(f"ion_gym v{ion_gym.__version__} · loaded from {_origin}")


<!-- origin-guard-note -->
The cell above only pins this notebook to its own repo's `ion_gym`. The notebook proper begins below.


In [ ]:
# ---- session resources: throughput choices, not physics -----------------
# The same defaults the workbench offers beside its Fly button. A deck
# never declares these, and a seeded flight returns the same results at
# any worker count.
from ion_gym.physics.ensemble_driver import default_workers, fly_all
from ion_gym.physics.sizing import default_record_budget_gb, resources_line

N_WORKERS = default_workers()      # ion-flight threads: every logical
                                   # core but two, so the kernel and the
                                   # OS stay responsive
RECORD_BUDGET_GB = default_record_budget_gb()   # GB the trajectory record
                                   # may use: RECORD_REFUSE_FRACTION of RAM,
                                   # the workbench default. build_run warns
                                   # at half of it and refuses above it.
                                   # None = RAM unreadable, fixed limits.
print(resources_line(N_WORKERS, RECORD_BUDGET_GB))

In [ ]:
# every choice in one place: device, operating point, ion count, figure
import sys
import time
from pathlib import Path

from IPython.display import Image, display

from ion_gym.io.paths import examples_dir, outputs_dir, repo_root

# the deck generator lives beside this notebook in the shipped
# manuscript_examples folder; banks and figures go to the outputs folder
STUDY = repo_root() / "manuscript_examples" / "drift_cell"
OUT = outputs_dir("manuscript_examples", "drift_cell", "run_export")
sys.path.insert(0, str(STUDY))
from make_rf_drift_cell import rf_drift_cell_spec   # noqa: E402

# ---- device: the 44-ring, 5 mm bore Allen and Bush cell of notebook 24 --
DEVICE = dict(
    bore_id_mm=5.0,       # ring inner diameter (mm)
    n_rings=44,           # 33 measured pitches plus entry and exit rings
    rf_freq_hz=2.8e6,     # RF frequency (Hz)
    p_torr=3.5,           # helium pressure of the measurements (Torr)
    T_k=298.0,            # gas temperature (K)
    gas="He",
    ion_mass_da=492.5,    # GRGDS [M+2H]2+ mass (Da), m/z 246.3
    ion_charge=2,
    sigma_m2=1.41e-18,    # hard-sphere cross section in He (m^2), 141 A^2
    source_ring=3,        # ions start half a pitch past ring 3
    exit_ring=41,         # flights end at the ring 41 centre plane
)
# ---- operating point: the middle of each campaign ladder ----------------
E_OVER_P = 0.507          # drift field (V cm^-1 Torr^-1), of 1.51, 0.507, 0.35
RF_VPP = 50.0             # RF amplitude (V peak to peak), of 0, 15, 50, 100, 200
DT_NS = 0.2               # step (ns), hard-sphere advisory dt <= tau_c / 20
SEED = 60601              # campaign seed, so the flight is reproducible
N_IONS = 30               # ions flown
# N_WORKERS and RECORD_BUDGET_GB come from the session-resources cell
# at the top, the same defaults the workbench offers beside its Fly button
T_MAX_US = None           # flight time limit (us); None takes the builder's
                          # rule, 3 x the Mason-Schamp drift time, printed
                          # in section 2
REC_INTERVAL_US = 3.2     # time between recorded trajectory points (us).
                          # The integrator steps at DT_NS; only every
                          # rec_every-th step is recorded, and panel A draws
                          # straight segments between records, so motion
                          # faster than this interval (RF micromotion, period
                          # 0.357 us) is not shown. Arrivals in panels B and
                          # C come from each ion's exact end state and do
                          # not depend on it.

# ---- figure -------------------------------------------------------------
ION_LABEL = "GRGDS [M+2H]$^{2+}$, 492.5 Da"   # named here: on the r-z
                          # route mz_list holds the ion MASS, so the
                          # figure is not left to label it as m/z
PLANE = "yz"              # cross-section plane of panel B
HIST_AXIS = "t"           # coordinate histogrammed in panel C
N_BINS = 15               # histogram bins in panel C (about 2 ions per
                          # bin at N_IONS = 30; raise with N_IONS)
N_PATHS = 12              # trajectories drawn in panel A (the first N_PATHS
                          # ions by index); only these are kept in the bank
SHOW_NOTE = True          # print the operating note under the figure
                          # (False for a journal version; the caption then
                          # states the operating point)

# ---- worker scaling (sections 7 to 9) ----------------------------------
import os                 # noqa: E402
N_SCALING_IONS = 48       # ions flown at every rung of the ladder; a
                          # multiple of the worker counts keeps the last
                          # batch full
WORKER_LADDER = sorted({w for w in (1, 2, 4, 8, 16, 32)
                        if w <= (os.cpu_count() or 1)} | {N_WORKERS})
BANK_PREFIX = "flight_"   # bank names are timestamped after this prefix
FIG_STEM = "drift_cell_run_export"

## 2. One timed ion sets the projected time of the batch

The deck is built once and one ion is flown untimed, because the first flight compiles the integrator. A second ion is then timed, and the projection below is that time multiplied by the number of ions and divided by the number of worker threads. The division assumes the ions share the cores evenly, so the projection is a lower bound. The batch runs in the next cell, so the decision to run it can be made here.

In [ ]:
# build the deck, time one warm ion, and print the projection
from ion_gym.physics.run_export import plan_flight
from ion_gym.progress import quote

REC_EVERY = max(1, round(REC_INTERVAL_US * 1e3 / DT_NS))
spec = rf_drift_cell_spec(**DEVICE, rf_vpp=RF_VPP, e_over_p=E_OVER_P,
                          n_ions=N_IONS, dt_ns=DT_NS, seed=SEED,
                          t_max_us=T_MAX_US, rec_every=REC_EVERY)
it = spec.integration
print(f"integration step {it.dt_ns:g} ns, one record every {it.rec_every} "
      f"steps ({it.rec_every * it.dt_ns * 1e-3:.3g} us), time limit "
      f"{it.t_max_us:.0f} us")
plan = plan_flight(spec, record_budget_gb=RECORD_BUDGET_GB)
quote("drift cell flight (one measured warm ion)",
      total_s=plan.projected_s(N_WORKERS),
      detail=f"{plan.n_ions} ions x {plan.per_ion_s:.2f} s / "
             f"{N_WORKERS} worker(s)")

## 3. The batch is flown and written to a bank

Every ion is flown on `N_WORKERS` threads with progress lines, and the arrivals on every impact surface are collected by `impact_hits`. The bank stores the deck itself, the arrivals, and the trajectories panel A draws, under a timestamped name that is never overwritten.

In [ ]:
# fly the batch, collect the arrivals, and bank the flight
from ion_gym.io.results_bank import bank_flight
from ion_gym.physics.run_export import fly_flight
results, hits = fly_flight(plan, label="drift cell", n_workers=N_WORKERS)
op = (f"Allen and Bush cell, {DEVICE['n_rings']} rings, E/P {E_OVER_P:g} "
      f"V/cm/Torr, {RF_VPP:g} Vpp, {DEVICE['gas']} {DEVICE['p_torr']:g} Torr, "
      f"{ION_LABEL}, {plan.n_ions} ions, seed {SEED}")
bank_path = bank_flight(
    OUT / (BANK_PREFIX + time.strftime("%Y%m%d_%H%M%S")), spec=spec,
    n_flown=plan.n_ions, cols=plan.cols, hits=hits,
    trajs=[r.traj for r in results[:N_PATHS]], operating_point=op)
print(f"banked {len(hits)} arrivals from {plan.n_ions} ions -> {bank_path}")

## 4. The figure is drawn from the newest bank

The figure reads only the bank, so it shows what was flown and can be redrawn at any time. The deck is rebuilt from the banked copy to draw panel A, and the exit plane is named from the deck's own bound.

In [ ]:
# load the newest bank and draw the three-panel print figure
from ion_gym.io.results_bank import latest_bank, load_flight_bank
from ion_gym.physics.sim_build import build_run
from ion_gym.viz.run_figures import publication_run_figure
from ion_gym.viz.viz_core import (OKABE_ITO, apply_house_style, save_figure,
                                  set_style)

apply_house_style()
set_style(groups=list(OKABE_ITO))
b_spec, b_n, b_cols, b_hits, b_trajs, b_meta = load_flight_bank(
    latest_bank(OUT, BANK_PREFIX))
surfaces = sorted({h["station"] for h in b_hits})
if len(surfaces) != 1:
    raise RuntimeError(f"expected arrivals on the exit plane only, found "
                       f"{surfaces}")
b_model = build_run(b_spec, record_budget_gb=RECORD_BUDGET_GB)[0]
fig = publication_run_figure(
    b_spec, b_model, b_trajs, b_hits, surface=surfaces[0], n_flown=b_n,
    plane=PLANE, hist_axis=HIST_AXIS, ion_label=ION_LABEL, n_bins=N_BINS,
    n_paths=N_PATHS, operating_note=SHOW_NOTE,
    surface_label=f"the exit plane (x = {b_spec.bounds.x_max:g} mm)")
for ext in ("png", "pdf"):
    save_figure(fig, OUT / f"{FIG_STEM}.{ext}")
display(Image(str(OUT / f"{FIG_STEM}.png")))

## 5. Transmission, arrival time, and radial spread are printed from the bank

In [ ]:
# the numbers a caption would quote, read from the same bank
import numpy as np

t = np.array([h["t_us"] for h in b_hits])
r = np.hypot([h["y"] for h in b_hits], [h["z"] for h in b_hits])
print(f"transmission to the exit plane: {len(b_hits)} of {b_n} ions")
# population standard deviation, ddof=0
print(f"arrival time from birth: mean {t.mean():.0f} us, "
      f"standard deviation {t.std(ddof=0):.0f} us, "
      f"range {t.min():.0f} to {t.max():.0f} us")
print(f"radius at the exit plane: max {r.max():.2f} mm, "
      f"rms {np.sqrt((r ** 2).mean()):.2f} mm "
      f"(bore radius {DEVICE['bore_id_mm'] / 2:g} mm)")

## 6. Reading the figure

The arrival times in panel C run from each ion's birth, half a pitch past ring 3, to the exit plane at ring 41. They are longer than the transit times of Figure 7 in the manuscript, which are measured over the 49.5 mm span between the ring 6 and ring 39 planes. Readers comparing the two should use the span stated with each figure. At this field and amplitude the campaign found full transmission, and the printed transmission above is the check of that expectation for this flight. A larger flight on a faster machine needs only a larger `N_IONS` in section 1, and the quote in section 2 states its time before it runs.

## 7. The wall time of a fixed batch is measured at each worker count

Ions fly independently in the absence of space charge, so the wall time of a batch should fall with the number of worker threads. The same seeded batch of `N_SCALING_IONS` ions is flown once at each count in `WORKER_LADDER`, and the machine is recorded with the timings. The projection below assumes ideal scaling, so it is a lower bound. The ladder runs in the next cell.

In [ ]:
# build the scaling batch, record the machine, and quote the whole ladder
from ion_gym.physics.run_export import machine_info

spec_s = rf_drift_cell_spec(**DEVICE, rf_vpp=RF_VPP, e_over_p=E_OVER_P,
                            n_ions=N_SCALING_IONS, dt_ns=DT_NS, seed=SEED,
                            t_max_us=T_MAX_US, rec_every=REC_EVERY)
plan_s = plan_flight(spec_s, record_budget_gb=RECORD_BUDGET_GB)
MACHINE = machine_info()
for k, v in MACHINE.items():
    print(f"  {k:18s} {v}")
quote("worker-scaling ladder (ideal scaling assumed)",
      total_s=sum(plan_s.projected_s(w) for w in WORKER_LADDER),
      detail=f"{N_SCALING_IONS} ions x {plan_s.per_ion_s:.2f} s at "
             f"workers {WORKER_LADDER}")

## 8. The ladder is flown and every rung is checked against the first

Each rung flies the identical seeded batch. Because each ion carries its own random seed, every rung must return the same arrivals, and the cell refuses to report timings if any rung differs. The timings and the machine are written to a bank.

In [ ]:
# fly the ladder, check identical arrivals, tabulate, and bank
from ion_gym.io.results_bank import bank_run

walls, ref = [], None
for w in WORKER_LADDER:
    t0 = time.perf_counter()
    _r, h_s = fly_flight(plan_s, label=f"{w} worker(s)", n_workers=w)
    walls.append(time.perf_counter() - t0)
    sig = sorted((x["t_us"], x["y"], x["z"]) for x in h_s)
    if ref is None:
        ref = sig
    elif sig != ref:
        raise RuntimeError(f"{w} workers returned different arrivals from "
                           f"{WORKER_LADDER[0]}: the seeded-ensemble contract "
                           f"is broken, so these timings are not comparable")
speedup = [walls[0] / t for t in walls]
print(f"{'workers':>8} {'wall (s)':>10} {'speedup':>8} {'efficiency':>11}")
for w, t, sp in zip(WORKER_LADDER, walls, speedup):
    print(f"{w:8d} {t:10.1f} {sp:8.2f} {sp / w:11.2f}")
print(f"identical arrivals at every rung: {len(ref)} ions")
SCALE_OP = (f"{N_SCALING_IONS} ions, {DEVICE['n_rings']}-ring drift cell, "
            f"E/P {E_OVER_P:g} V/cm/Torr, {RF_VPP:g} Vpp, dt {DT_NS:g} ns; "
            f"{MACHINE['cpu']}, {MACHINE['logical_cores']} logical cores")
scale_bank = bank_run(
    OUT / ("scaling_" + time.strftime("%Y%m%d_%H%M%S")),
    meta=dict(operating_point=SCALE_OP, machine=MACHINE,
              n_ions=N_SCALING_IONS, seed=SEED),
    arrays=dict(workers=np.array(WORKER_LADDER), wall_s=np.array(walls)))
print(f"banked -> {scale_bank}")

## 9. Speedup against worker count, with the ideal line for reference

In [ ]:
# the scaling figure, through the framework's profile renderer
from ion_gym.viz.design_maps import ProfileCurve, render_profiles

fig_s = render_profiles(
    [ProfileCurve("measured", WORKER_LADDER, speedup,
                  {"marker": "o", "color": OKABE_ITO[0]}),
     ProfileCurve("ideal", WORKER_LADDER, WORKER_LADDER,
                  {"ls": "--", "color": "0.5"})],
    xlabel="$N$ (worker threads)", ylabel="$t_1 / t_N$",
    title="Wall time of a fixed batch falls with the number of workers",
    subtitle=SCALE_OP)
# worker counts are whole numbers: no fractional ticks on the x axis
from matplotlib.ticker import MaxNLocator
fig_s.axes[0].xaxis.set_major_locator(MaxNLocator(integer=True))
fig_s.savefig(OUT / "worker_scaling.png", dpi=300)
display(Image(str(OUT / "worker_scaling.png")))
n_max = WORKER_LADDER[-1]
print(f"On {MACHINE['cpu']} ({MACHINE['logical_cores']} logical cores), "
      f"{N_SCALING_IONS} ions of the {DEVICE['n_rings']}-ring drift cell deck "
      f"flew in {walls[0]:.0f} s on 1 worker and {walls[-1]:.0f} s on "
      f"{n_max} workers, a speedup of {speedup[-1]:.1f}.")

The printed sentence states the measured scaling with its machine, worker count, and deck, the three items the manuscript statement on parallel workers needs. On a machine with performance and efficiency cores (Apple silicon, recent Intel), the ladder passes from one kind to the other as the worker count rises, and the efficiency column shows where. The figure and the bank keep the measurement with its operating point.